In [ ]:
# Import libraries for Windows log collection, preprocessing and LSTM model

import subprocess
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Collect Windows Security events only from the last hour

import os
import subprocess
import pandas as pd

os.makedirs("data", exist_ok=True)

# Use raw string for Windows path
csv_path = r".\data\security_events.csv"

powershell_command = f"""
$startTime = (Get-Date).AddHours(-1)

Get-WinEvent -FilterHashtable @{{LogName='Security'; StartTime=$startTime}} -ErrorAction Stop |
Select-Object TimeCreated, Id, ProviderName, MachineName, Message |
Export-Csv "{csv_path}" -NoTypeInformation -Encoding UTF8
"""

result = subprocess.run(
    ["powershell", "-ExecutionPolicy", "Bypass", "-Command", powershell_command],
    capture_output=True,
    text=True
)

print("PowerShell return code:", result.returncode)

if result.stdout:
    print("STDOUT:", result.stdout)

if result.stderr:
    print("STDERR:", result.stderr)

if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
    raise Exception("CSV file was not created or is empty. Check PowerShell error above.")

df = pd.read_csv(csv_path)

df["TimeCreated"] = pd.to_datetime(df["TimeCreated"], errors="coerce")
df = df.dropna(subset=["TimeCreated"])
df = df.sort_values("TimeCreated")

print("Collected events:", len(df))

df.head()

In [ ]:
# Collect Task Scheduler operational events from the last hour

task_csv_path = r".\data\task_scheduler_events.csv"

powershell_command = f"""
$startTime = (Get-Date).AddHours(-1)

Get-WinEvent -FilterHashtable @{{LogName='Microsoft-Windows-TaskScheduler/Operational'; StartTime=$startTime}} -ErrorAction Stop |
Select-Object TimeCreated, Id, ProviderName, MachineName, Message |
Export-Csv "{task_csv_path}" -NoTypeInformation -Encoding UTF8
"""

result = subprocess.run(
    ["powershell", "-ExecutionPolicy", "Bypass", "-Command", powershell_command],
    capture_output=True,
    text=True
)

print("PowerShell return code:", result.returncode)
print(result.stderr)

task_df = pd.read_csv(task_csv_path)
task_df["TimeCreated"] = pd.to_datetime(task_df["TimeCreated"], errors="coerce")
task_df = task_df.dropna(subset=["TimeCreated"])
task_df = task_df.sort_values("TimeCreated")

task_df.tail(20)

In [ ]:
# Combine Security logs and Task Scheduler logs

df = pd.concat([df, task_df], ignore_index=True)
df = df.sort_values("TimeCreated")

print("Combined events:", len(df))
df.tail(30)

In [ ]:
# Convert raw Windows events into abstract security behavior events

def map_event(row):

    event_id = int(row["Id"])
    message = str(row["Message"]).lower()

    # Task Scheduler operational events
    if "taskscheduler" in str(row["ProviderName"]).lower():
        if "labdemotask" in message:
            return "TASK_CREATED"
        if event_id in [106, 140, 200, 201]:
            return "TASK_SCHEDULER_ACTIVITY"

    # Authentication events
    if event_id == 4624:
        return "LOGIN_SUCCESS"

    if event_id == 4625:
        return "LOGIN_FAILED"

    if event_id == 4634:
        return "LOGOFF"

    if event_id == 4740:
        return "ACCOUNT_LOCKED"

    # Process creation
    if event_id == 4688:

        if "powershell" in message:

            if "-enc" in message or "encodedcommand" in message:
                return "ENCODED_POWERSHELL"

            return "POWERSHELL_EXECUTION"

        if "cmd.exe" in message:
            return "CMD_EXECUTION"

        if "schtasks" in message:
            return "TASK_COMMAND"

        return "PROCESS_CREATED"

    # Scheduled task creation
    if event_id == 4698:
        return "TASK_CREATED"

    # Network share access
    if event_id == 5140:
        return "SMB_SHARE_ACCESS"

    # Group membership changes
    if event_id in [4728, 4732, 4756]:
        return "GROUP_MEMBERSHIP_CHANGED"

    # User account created
    if event_id == 4720:
        return "USER_CREATED"

    # User account deleted
    if event_id == 4726:
        return "USER_DELETED"

    # Service installed
    if event_id == 4697:
        return "SERVICE_INSTALLED"

    # Default fallback
    return "OTHER_EVENT"

In [ ]:
# Apply abstraction mapping and show abstract event distribution

df["abstract_event"] = df.apply(map_event, axis=1)

event_counts = df["abstract_event"].value_counts()

print(event_counts)

event_counts.plot(kind="bar")
plt.title("Abstract Windows Event Distribution")
plt.xlabel("Event Type")
plt.ylabel("Count")
plt.show()

In [ ]:
# Encode abstract events into numeric tokens

event_vocab = {
    event: index + 1
    for index, event in enumerate(sorted(df["abstract_event"].unique()))
}

reverse_vocab = {
    value: key
    for key, value in event_vocab.items()
}

df["event_token"] = df["abstract_event"].map(event_vocab)

print(event_vocab)

In [ ]:
# Build event sequences from real Windows logs

sequence_length = 10

tokens = df["event_token"].tolist()
events = df["abstract_event"].tolist()

X = []
y = []

for i in range(len(tokens) - sequence_length):
    sequence_tokens = tokens[i:i + sequence_length]
    sequence_events = events[i:i + sequence_length]

    X.append(sequence_tokens)

    # Rule-based label for training demonstration:
    # suspicious if the sequence contains task creation or repeated login failures
    suspicious = (
        "TASK_CREATED" in sequence_events
        or "TASK_COMMAND" in sequence_events
        or sequence_events.count("LOGIN_FAILED") >= 3
    )

    y.append(1 if suspicious else 0)

X = np.array(X)
y = np.array(y)

print("Total sequences:", X.shape[0])
print("Suspicious sequences:", y.sum())
print("Normal sequences:", len(y) - y.sum())

In [ ]:
# If there are no suspicious sequences yet, the model cannot learn both classes

if len(set(y)) < 2:
    print("Only one class found in current logs.")
    print("Run a benign scheduled task creation demo on your lab VM, then collect logs again.")
else:
    print("Both normal and suspicious classes are available.")

In [ ]:
# Train LSTM model only if both classes exist

if len(set(y)) >= 2:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    vocab_size = len(event_vocab) + 1

    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=16),
        LSTM(32),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()

    history = model.fit(
        X_train,
        y_train,
        epochs=5,
        batch_size=16,
        validation_data=(X_test, y_test)
    )

In [ ]:
# Evaluate model performance

if len(set(y)) >= 2:
    loss, accuracy = model.evaluate(X_test, y_test)

    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Find and analyze the latest sequence that contains TASK_CREATED

target_token = event_vocab.get("TASK_CREATED")

if target_token is None:
    print("TASK_CREATED was not found in the current vocabulary.")
else:
    matching_indexes = [
        i for i, sequence in enumerate(X)
        if target_token in sequence
    ]

    print("Sequences with TASK_CREATED:", len(matching_indexes))

    if matching_indexes:
        idx = matching_indexes[-1]

        selected_sequence = X[idx].reshape(1, -1)

        decoded_sequence = [
            reverse_vocab[token]
            for token in X[idx]
        ]

        print("Selected event sequence:")
        print(" → ".join(decoded_sequence))

        probability = model.predict(selected_sequence)[0][0]

        print(f"Suspicious probability: {probability * 100:.2f}%")

        if probability > 0.5:
            print("ALERT: Suspicious behavior pattern detected")
        else:
            print("Normal behavior")

In [ ]:
# Analyze the latest sequence from real Windows logs

latest_sequence = X[-1].reshape(1, -1)

decoded_sequence = [
    reverse_vocab[token]
    for token in X[-1]
]

print("Latest event sequence:")
print(" → ".join(decoded_sequence))

if len(set(y)) >= 2:
    probability = model.predict(latest_sequence)[0][0]

    print(f"Suspicious probability: {probability * 100:.2f}%")

    if probability > 0.5:
        print("ALERT: Suspicious behavior pattern detected")
    else:
        print("Normal behavior")
else:
    print("Model was not trained because only one class exists in current logs.")